# Preprocess & Merge (Reproducible Pipeline)
This notebook shows how we merged the base dataset with DIU-collected records, cleaned columns, normalized scales, and saved the extended dataset.

In [ ]:
import pandas as pd
import numpy as np

# Paths (replace with your local files)
base_path = 'base_public_dataset.csv'           # Syeed et al. 2024 (after download)
diu_new_path = 'diu_new_samples.csv'            # DIU-collected samples
out_path = 'data/DIU_Extended_Mental_Health.csv'

# 1) Load
base = pd.read_csv(base_path)
diu = pd.read_csv(diu_new_path)

# 2) Harmonize column names/types
def normalize(df):
    df = df.copy()
    # example cleanup
    df['Academic_Year'] = df['Academic_Year'].astype(str)
    return df

base = normalize(base)
diu = normalize(diu)

# 3) Concatenate
combined = pd.concat([base, diu], ignore_index=True)

# 4) Optional: clip scales to [1..4]
for p in [*[f'GAD{i}' for i in range(1,8)], *[f'PSS{i}' for i in range(1,11)], *[f'DASS_D{i}' for i in range(1,8)]]:
    if p in combined.columns:
        combined[p] = combined[p].clip(1,4)

# 5) Save (ensure final column ordering)
cols = ['Age', 'Current_CGPA', 'Gender', 'University', 'Department', 'Academic_Year', 'waiver_or_scholarship', 'GAD1', 'GAD2', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PSS1', 'PSS2', 'PSS3', 'PSS4', 'PSS5', 'PSS6', 'PSS7', 'PSS8', 'PSS9', 'PSS10', 'DASS_D1', 'DASS_D2', 'DASS_D3', 'DASS_D4', 'DASS_D5', 'DASS_D6', 'DASS_D7', 'Anxiety Label', 'Stress Label', 'Depression Label']
for c in cols:
    if c not in combined.columns:
        combined[c] = np.nan
combined = combined[cols]
combined.to_csv(out_path, index=False)
print('Saved:', out_path)
